In [11]:
import os
import shutil

def list_files_as_set(path):
    files = os.listdir(path)
    return set(files)

path_current_results = "./sim_results/"
path_laptop_results = "./sim_results_laptop/sim_results"
path_old_optimal_results = "/home/mateus/NFR-master/Simulator/sim_results/optimal_solutions/"
path_current_optimal_results = path_current_results + "optimal_solutions/"

current_results_files = list_files_as_set(path_current_results)
laptop_results_files = list_files_as_set(path_laptop_results)

old_optimal_results_files = list_files_as_set(path_old_optimal_results)
current_optimal_results_files = list_files_as_set(path_current_optimal_results)



In [14]:
# Copying files 
# import shutil
# diff = laptop_results_files - current_results_files
# diff

# for file in diff:
    # shutil.copy(path_laptop_results + "/" + file, path_current_results)

In [15]:
import json
import os
import re
import numpy as np


def compare_same_sim_scenario(cplex_path, scipy_path, atol=1e-5):
    with open(cplex_path) as f:
        cplex = json.load(f)

    with open(scipy_path) as f:
        scipy = json.load(f)

    filename = os.path.basename(scipy_path)
    C = int(re.search(r"_C(\d+)_", filename).group(1))

    pi_bs = np.asarray(cplex["pi_bs"])

    # CPtop -> same top-C cache used by the optimization
    cache = np.argsort(pi_bs)[-C:]

    pi_c = np.asarray(cplex["pi_final_NA"])
    pi_s = np.asarray(scipy["pi_final_NA"])

    # LP objective: probability of requesting an uncached item
    obj_c = 1.0 - pi_c[cache].sum()
    obj_s = 1.0 - pi_s[cache].sum()

    return np.isclose(obj_c, obj_s, atol=atol, rtol=0)

In [17]:
intersection = old_optimal_results_files & current_optimal_results_files

matches = sum(
    compare_same_sim_scenario(
        path_old_optimal_results + file,
        path_current_optimal_results + file
    )
    for file in intersection
)

print(f"{matches}/{len(intersection)} match")

37/50 match
